# 📓 Semana 4 · Dia 2 — Delta: MERGE, schema evolution e constraints

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | DEA (Delta) |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | MERGE com schema evolution rodando |

---


## 📖 Teoria — O MERGE (upsert)

**MERGE** insere, atualiza ou apaga linhas em uma única operação atômica: 

```sql
MERGE INTO alvo t
USING origem s ON t.id = s.id
WHEN MATCHED THEN UPDATE SET ...
WHEN NOT MATCHED THEN INSERT ...
```

É o padrão para: deduplicar, SCD, cargas incrementais idempotentes. Sem MERGE, você precisaria de delete+insert não atômicos.


## 📖 Teoria — Schema evolution

Por padrão, uma escrita com colunas novas **falha** (enforcement). Com `mergeSchema = True` (ou `ALTER TABLE ADD COLUMN`), o schema evolui adicionando colunas. Use com cuidado: evolução automática pode quebrar consumidores.


### 💻 Na prática — MERGE na prática

Monte uma tabela alvo e uma fonte com updates/inserts, e aplique MERGE.


In [ ]:
# Alvo: dimensão de clientes (simulada)
spark.sql("CREATE OR REPLACE TABLE workspace.prata.dim_cliente_teste (
  CustomerID STRING, nome STRING, cidade STRING, is_current BOOLEAN) USING DELTA")
spark.sql("INSERT INTO workspace.prata.dim_cliente_teste VALUES
  ('12345', 'Ana', 'SP', true),
  ('67890', 'João', 'RJ', true)")
print("Alvo inicial criado")

In [ ]:
# Fonte: clientes novos + Ana mudou de cidade
fonte = spark.createDataFrame([
    ("12345", "Ana", "CAMPINAS"),
    ("99999", "Maria", "BH"),
], ["CustomerID", "nome", "cidade"])
fonte.createOrReplaceTempView("fonte_cli")
print("Fonte com 1 update + 1 insert")

In [ ]:
%sql
-- MERGE: atualiza Ana, insere Maria
MERGE INTO workspace.prata.dim_cliente_teste t
USING fonte_cli s ON t.CustomerID = s.CustomerID
WHEN MATCHED THEN UPDATE SET t.cidade = s.cidade
WHEN NOT MATCHED THEN INSERT (CustomerID, nome, cidade, is_current)
  VALUES (s.CustomerID, s.nome, s.cidade, true);
SELECT * FROM workspace.prata.dim_cliente_teste ORDER BY CustomerID;

### 💻 Na prática — Schema evolution

Adicione uma coluna nova na fonte e evolua o schema do alvo.


In [ ]:
# Fonte com coluna nova (email)
fonte2 = spark.createDataFrame([("12345", "Ana", "CAMPINAS", "ana@x.com")],
                               ["CustomerID", "nome", "cidade", "email"])
fonte2.createOrReplaceTempView("fonte2")
print("Fonte agora tem coluna email")

In [ ]:
%sql
-- MERGE com schema evolution (novo campo email)
MERGE INTO workspace.prata.dim_cliente_teste t
USING fonte2 s ON t.CustomerID = s.CustomerID
WHEN MATCHED THEN UPDATE SET t.cidade = s.cidade
WHEN NOT MATCHED THEN INSERT (CustomerID, nome, cidade, is_current)
  VALUES (s.CustomerID, s.nome, s.cidade, true);
-- Falha por padrão (enforcement)!
-- Para permitir evolução, use spark.conf ou ALTER TABLE:

In [ ]:
# Habilitar evolução automática e refazer
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
spark.sql("""
MERGE INTO workspace.prata.dim_cliente_teste t
USING fonte2 s ON t.CustomerID = s.CustomerID
WHEN MATCHED THEN UPDATE SET t.cidade = s.cidade, t.email = s.email
WHEN NOT MATCHED THEN INSERT *
""")
display(spark.sql("SELECT * FROM workspace.prata.dim_cliente_teste"))

> 🎯 **Dica de prova**: MERGE com `WHEN NOT MATCHED THEN INSERT *` + `autoMerge` evolui schema automaticamente. Pergunta típica: o que acontece sem schema evolution? → falha de análise.


## 🎯 Exercícios de fixação

**1.** Escreva um MERGE que faça UPDATE apenas quando a cidade mudou (evita escrita desnecessária).

**2.** O que `INSERT *` faz no MERGE?

**3.** Crie um SCD1 simples com MERGE (update de cidade sem histórico).


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** MERGE condicional

```sql
WHEN MATCHED AND t.cidade <> s.cidade THEN UPDATE SET t.cidade = s.cidade
```

**2.** INSERT *

Insere todas as colunas da fonte que não estão na condição de match — com autoMerge, cria as colunas novas automaticamente.

**3.** SCD1

O MERGE acima (update direto) É o SCD1: sobrescreve o valor antigo, sem histórico.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*